In [2]:
!pip install sentence-transformers faiss-cpu google-generativeai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 80.8 MB/s eta 0:00:00


In [3]:
import pandas as pd
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer

import google.generativeai as genai

from google.colab import userdata

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [5]:
GOOGLE_API_KEY = userdata.get(
    "Research++"
)

genai.configure(
    api_key=GOOGLE_API_KEY
)

llm = genai.GenerativeModel(
    "gemini-2.5-flash"
)

print("Gemini configured successfully.")

Gemini configured successfully.


In [10]:
from google.colab import files

uploaded = files.upload()
uploaded = files.upload()

Saving researchforge_faiss.index to researchforge_faiss (2).index


Saving papers.json to papers.json


In [11]:
df = pd.read_json(
    "papers.json"
)

faiss_index = faiss.read_index(
    "researchforge_faiss.index"
)

print(
    f"Loaded {len(df)} papers."
)

print(
    f"FAISS vectors: {faiss_index.ntotal}"
)

Loaded 100 papers.
FAISS vectors: 100


In [12]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print(
    "Embedding model ready."
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model ready.


In [13]:
def retrieve_papers(
    user_query,
    top_k=5
):

    query_embedding = embedding_model.encode(
        [user_query]
    )

    distances, indices = faiss_index.search(
        np.array(
            query_embedding
        ).astype("float32"),
        k=top_k
    )

    retrieved_papers = []

    for idx in indices[0]:

        retrieved_papers.append({

            "title":
            df.iloc[idx]["title"],

            "abstract":
            df.iloc[idx]["abstract"],

            "published":
            df.iloc[idx]["published"]

        })

    return retrieved_papers

In [14]:
query = """
Find recent methods for
multimodal deepfake detection
using transformers
"""

results = retrieve_papers(
    query
)

for paper in results:

    print("=" * 90)

    print(
        paper["title"]
    )

    print()

Deepfake Synthesis vs. Detection: An Uneven Contest

Leveraging large multimodal models for audio-video deepfake detection: a pilot study

Integrating Audio-Visual Features for Multimodal Deepfake Detection

WWW: Where, Which and Whatever Enhancing Interpretability in Multimodal Deepfake Detection

DeepFake-Adapter: Dual-Level Adapter for DeepFake Detection



In [15]:
def generate_research_summary(
    user_query,
    retrieved_papers
):

    evidence_context = ""

    for paper in retrieved_papers:

        evidence_context += f"""

Title:
{paper['title']}

Abstract:
{paper['abstract']}

"""

    prompt = f"""

You are an AI scientific research assistant.

User Question:

{user_query}

Retrieved Evidence:

{evidence_context}

Task:

1. Summarize key methods.

2. Compare approaches.

3. Mention limitations.

4. Ground claims in retrieved evidence.

"""

    response = llm.generate_content(
        prompt
    )

    return response.text

In [16]:
query = """
What are the major research trends
in multimodal deepfake detection?
"""

papers = retrieve_papers(
    query
)

answer = generate_research_summary(
    query,
    papers
)

print(answer)

Based on the retrieved evidence, the major research trends in multimodal deepfake detection encompass the evolution of detection methodologies, a strong emphasis on comprehensive dataset development, and the leveraging of advanced AI models.

Here are the key trends, comparisons, and limitations:

**1. Key Methods:**

*   **Traditional Unimodal and Ensemble-Based Detectors:** Initial approaches focused on detecting deepfakes in single modalities (either video or audio) or combining these unimodal detectors through ensemble methods (Abstract 1, 5).
*   **Feature Integration Methods:** Researchers are developing methods that explicitly integrate audio-visual features. One approach categorizes samples by combining labels from single modalities to enhance detection (Abstract 5).
*   **Large Multimodal Models (LMMs):** A significant trend involves leveraging large, pre-trained multimodal models (LMMs). For instance, AV-LMMDetect utilizes a supervised fine-tuned LMM built on models like Qwen